In [1]:
import sys
sys.path.append('../utils_function/')
sys.path.append(r'/data/zyjin/common_func/')
import utils_function as uf
import common_functions as cf
import pandas as pd
import numpy as np
import os
from collections import defaultdict

In [2]:
def get_optimal_params(performance_dict):
    """
    从性能字典中找出平均相关性最高的参数
    
    参数:
    performance_dict (dict): 包含不同参数对应相关性列表的字典
    
    返回:
    float: 最优参数值
    """
    best_lambda = None
    best_avg_corr = -float('inf')  # 初始化为负无穷

    # 遍历每个 lambda 及其对应的相关性列表
    for lambda_val, corr_list in performance_dict.items():
        # 计算当前 lambda 的平均相关性
        avg_corr = sum(corr_list) / len(corr_list)
        
        # 更新最优参数
        if avg_corr > best_avg_corr:
            best_avg_corr = avg_corr
            best_lambda = lambda_val
    
    return best_lambda

In [3]:
animal_list = ['mouse', 'marmoset', 'macaque', 'human', 'marmoset']
dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere', 'V4']
performance_dict = {}
optimal_params_dict = {}
for animal, dataset_key in zip(animal_list, dataset_key_list):
    print(f'Running for {animal} and {dataset_key}')

    performance_dict_this_animal = defaultdict(list)

    model_container = cf.MetaModelContainer(uf.AxonGrowthModel, cf.pj('../../results/para_search', f'{animal}_{dataset_key}'))

    for model in model_container:
        performance_dict_this_animal[model.get_unique_Lambda()].append(model.get_data_model_correlation())
    
    performance_dict[f'{animal}_{dataset_key}'] = performance_dict_this_animal
    optimal_params_dict[f'{animal}_{dataset_key}'] = {'Lambda': get_optimal_params(performance_dict_this_animal)}
    print(performance_dict_this_animal)
    print(f'Optimal lambda for {animal}_{dataset_key}: {optimal_params_dict[f"{animal}_{dataset_key}"]}')
    print(f'Optimal average correlation: {max([sum(v)/len(v) for v in performance_dict_this_animal.values()])}')


cf.save_dict(optimal_params_dict, '../../results/optimal_params/optimal_params')
cf.save_dict(performance_dict, '../../results/optimal_params/performance_dict')

Running for mouse and gamanut
defaultdict(<class 'list'>, {0.75: [0.8715547264299531, 0.871225892685422, 0.8714291801223926, 0.8715855199919339, 0.8715387629089082], 0.8500000000000001: [0.8723696048058639, 0.8724504500744078, 0.8717349725246489, 0.8720891526106325, 0.8721397653724237], 0.8999999999999999: [0.8721362964597705, 0.871131745932154, 0.8717374344573428, 0.8713554620653126, 0.8710713950923494], 0.725: [0.8713825699100658, 0.8714132205227869, 0.8714120048053273, 0.8722249435947573, 0.8716134852048139], 1.0999999999999999: [0.8684059825682193, 0.8691947866501049, 0.8698928299980854, 0.8693964678395852, 0.8697437049884681], 1.2999999999999998: [0.866195810407437, 0.8656848663255986, 0.8658358477870656, 0.8652665057366252, 0.8659859230013309], 0.9999999999999999: [0.870655241027811, 0.8699896108546917, 0.8707465169035363, 0.8708019022905832, 0.8706230921448392], 1.1999999999999997: [0.8682209014378132, 0.8679020772915622, 0.8680997601162646, 0.8677599346850008, 0.867386387141931